# Fare visualizations
Figures for the midterm report. Fares are rate-card estimates until the regression is fit on collected quotes; then set `FARE = "predicted_fare"` and load `tract_airport_fares.csv` instead.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import AIRPORTS, COUNTY_FIPS, CRS, OUTPUT_DIR, PROCESSED_DIR, RAW_DIR
from src.fares import rate_card_fare
from src.lodes import download
from src.routing import TERMINALS

In [ ]:
COUNTY_NAMES = {
    "001": "Alameda",
    "013": "Contra Costa",
    "041": "Marin",
    "055": "Napa",
    "075": "San Francisco",
    "081": "San Mateo",
    "085": "Santa Clara",
    "095": "Solano",
    "097": "Sonoma",
}
AIRPORT_COLORS = {"SFO": "#003262", "OAK": "#FDB515", "SJC": "#3B7EA1"}
FARE = "rate_card_fare"
FARE_LABEL = "rate-card estimate, before surge"
# LODES' lowest earnings band tops out at $1,250/month; over 260 workdays a year that is about $58/day
LOW_WAGE_DAILY = 1250 * 12 / 260

### Load data

In [ ]:
def load_tracts():
    # cartographic boundaries are clipped to the shoreline, so the bay reads as water on the maps
    path = f"{RAW_DIR}/ca_tracts_cb_2020.zip"
    download("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_06_tract_500k.zip", path)
    tracts = gpd.read_file(path)
    tracts = tracts[tracts["COUNTYFP"].isin(COUNTY_FIPS)]
    return tracts[["GEOID", "COUNTYFP", "geometry"]].to_crs(CRS)


tracts = load_tracts()
counties = tracts.dissolve("COUNTYFP")
terminals = gpd.GeoDataFrame(
    {"code": list(TERMINALS)},
    geometry=gpd.points_from_xy([lon for _, lon in TERMINALS.values()], [lat for lat, _ in TERMINALS.values()]),
    crs=4326,
).to_crs(CRS)

fares = rate_card_fare(pd.read_csv(f"{PROCESSED_DIR}/tract_airport_routes.csv", dtype={"GEOID": str}, keep_default_na=False))
fares["county"] = fares["GEOID"].str[2:5].map(COUNTY_NAMES)
workers = pd.read_csv(f"{PROCESSED_DIR}/airport_workers_by_tract.csv", dtype={"GEOID": str})
# inner join drops workers in the few tracts with no residents in the 2020 census, so no centroid
worker_fares = workers.merge(fares, on=["code", "GEOID"])
quotes = pd.read_csv(f"{PROCESSED_DIR}/fare_quotes.csv", dtype={"GEOID": str})
print(f"tracts: {len(tracts):,}  routes: {len(fares):,}  workers matched: {worker_fares['S000'].sum():,} of {workers['S000'].sum():,}")

In [ ]:
def weighted_median(values, weights):
    order = np.argsort(values.to_numpy())
    values, weights = values.to_numpy()[order], weights.to_numpy()[order]
    return values[np.searchsorted(np.cumsum(weights), weights.sum() / 2)]

### Estimated fare from every tract

In [ ]:
def plot_fare_maps():
    gdf = tracts.merge(fares, on="GEOID")
    vmax = gdf[FARE].quantile(0.95)
    fig, axes = plt.subplots(1, 3, figsize=(13, 5.8))

    for ax, airport in zip(axes, AIRPORTS):
        trips = gdf[gdf["code"] == airport]
        trips.plot(column=FARE, ax=ax, cmap="YlGnBu", vmin=0, vmax=vmax, linewidth=0)
        counties.boundary.plot(ax=ax, color="white", linewidth=0.6)
        terminals[terminals["code"] == airport].plot(ax=ax, marker="*", color="#FDB515", edgecolor="black", markersize=300, zorder=3)
        ax.set_title(f"To {airport} (median tract ${trips[FARE].median():.0f})", fontsize=11)
        ax.set_axis_off()

    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.17, wspace=0.02)
    cax = fig.add_axes([0.3, 0.11, 0.4, 0.025])
    bar = fig.colorbar(plt.cm.ScalarMappable(cmap="YlGnBu", norm=plt.Normalize(0, vmax)), cax=cax, orientation="horizontal", extend="max")
    bar.set_label("One-way fare ($)")
    fig.suptitle("A rideshare to the airport costs under \\$40 only for tracts near the terminal; the North Bay pays over \\$100", fontsize=12)
    caption = f"One-way fare from each tract's population-weighted centroid to the terminal ({FARE_LABEL}, incl. airport fee and tolls; star = terminal)"
    fig.text(0.5, 0.02, caption, ha="center", fontsize=9)
    fig.savefig(f"{OUTPUT_DIR}/fare_maps.png", dpi=300)
    return fig


fig = plot_fare_maps()

### What airport workers would pay

In [ ]:
def plot_worker_fare_distribution():
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    breakeven = LOW_WAGE_DAILY / 2

    for airport in AIRPORTS:
        trips = worker_fares[worker_fares["code"] == airport].sort_values(FARE)
        share = trips["S000"].cumsum() / trips["S000"].sum() * 100
        median = weighted_median(trips[FARE], trips["S000"])
        ax.step(trips[FARE], share, where="post", color=AIRPORT_COLORS[airport], linewidth=2, label=f"{airport} (median worker ${median:.0f})")

    ax.axvline(breakeven, color="gray", linestyle="--", linewidth=1)
    ax.text(breakeven + 2, 5, f"${breakeven:.0f}: round trip = a full day's pay\nfor the lowest earnings band (<= $1,250/mo)", fontsize=8.5, color="dimgray")
    ax.set_xlabel("One-way fare from home tract ($)")
    ax.set_ylabel("% of airport's Bay Area workers")
    ax.set_xlim(0, worker_fares[FARE].quantile(0.99))
    ax.set_ylim(0, 100)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="lower right")

    fig.suptitle("For most SFO and OAK workers, a rideshare round trip costs more than a low-wage day's pay", fontsize=12)
    caption = f"Cumulative share of workers by one-way fare from their home tract ({FARE_LABEL}; 2023 LODES workers)"
    fig.text(0.5, 0.01, caption, ha="center", fontsize=9)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"{OUTPUT_DIR}/worker_fare_distribution.png", dpi=300)
    return fig


fig = plot_worker_fare_distribution()

In [ ]:
def worker_fare_summary():
    """Worker-weighted fare stats per airport, for quoting in the report text."""
    rows = []
    for airport, trips in worker_fares.groupby("code"):
        rows.append({
            "airport": airport,
            "workers": trips["S000"].sum(),
            "median_one_way": weighted_median(trips[FARE], trips["S000"]),
            "median_low_wage_one_way": weighted_median(trips[FARE], trips["SE01"]),
            "pct_round_trip_over_low_wage_day": (trips.loc[2 * trips[FARE] > LOW_WAGE_DAILY, "S000"].sum() / trips["S000"].sum() * 100),
            "pct_paying_toll": trips.loc[trips["toll"] > 0, "S000"].sum() / trips["S000"].sum() * 100,
        })
    return pd.DataFrame(rows).set_index("airport").round(1)


worker_fare_summary()

### Where the money goes

In [ ]:
def plot_fare_components():
    components = {"ride_charge": ("Distance and time", "#003262"), "service_fee": ("Service fee", "#3B7EA1"), "airport_fee": ("Airport fee", "#FDB515"), "toll": ("Bridge toll", "#C4820E")}
    weighted = worker_fares[list(components)].mul(worker_fares["S000"], axis=0)
    weighted[["code", "county", "S000"]] = worker_fares[["code", "county", "S000"]]
    sums = weighted.groupby(["code", "county"]).sum()
    # worker-weighted mean of each component by home county
    means = sums[list(components)].div(sums["S000"], axis=0)
    order = means.loc["SFO"].sum(axis=1).sort_values().index

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), sharey=True, sharex=True)
    for ax, airport in zip(axes, AIRPORTS):
        table = means.loc[airport].reindex(order).fillna(0)
        left = np.zeros(len(table))
        for column, (label, color) in components.items():
            ax.barh(table.index, table[column], left=left, color=color, label=label)
            left += table[column]
        for county, total in zip(table.index, left):
            ax.text(total + 2, county, f"${total:.0f}", va="center", fontsize=8.5)
        ax.set_title(f"To {airport}", fontsize=11)
        ax.set_xlabel("Mean one-way fare per worker ($)")
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].legend(frameon=False, loc="lower right", fontsize=8.5)
    fig.suptitle("Fixed fees are a quarter to a third of short trips' cost; distance dominates long ones", fontsize=12)
    caption = f"Worker-weighted mean fare components by home county ({FARE_LABEL}; 2023 LODES workers)"
    fig.text(0.5, 0.01, caption, ha="center", fontsize=9)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"{OUTPUT_DIR}/fare_components_by_county.png", dpi=300)
    return fig


fig = plot_fare_components()

### Fare against trip distance
Collected quotes are overlaid once `fare_quotes.csv` has prices, as a first check of the rate card against real upfront prices.

In [ ]:
def plot_fare_vs_distance():
    kinds = {"No toll": "#003262", "State bridge ($8.50)": "#FDB515", "Golden Gate ($10.25)": "#EE1F60"}
    kind = np.select([fares["bridges"].str.contains("golden_gate"), fares["toll"] > 0], ["Golden Gate ($10.25)", "State bridge ($8.50)"], "No toll")
    collected = quotes.dropna(subset=["price"])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.3), sharey=True, sharex=True)
    for ax, airport in zip(axes, AIRPORTS):
        for label, color in kinds.items():
            trips = fares[(fares["code"] == airport) & (kind == label)]
            ax.scatter(trips["miles"], trips[FARE], s=6, alpha=0.5, color=color, label=label)
        ax_quotes = collected[collected["code"] == airport]
        if len(ax_quotes):
            ax.scatter(ax_quotes["miles"], ax_quotes["price"], s=25, marker="x", color="black", label="Collected quote")
        ax.set_title(f"To {airport}", fontsize=11)
        ax.set_xlabel("Route distance (miles)")
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel("One-way fare ($)")
    axes[0].legend(frameon=False, markerscale=2.5, fontsize=8.5)
    fig.suptitle("Rate-card fares are linear in distance, so which region's rate card applies matters as much as tolls", fontsize=12)
    caption = f"Each point is one tract-to-airport trip ({FARE_LABEL}). The lower line is Santa Clara origins priced on Lyft's Silicon Valley card (\\$0.84/mi vs \\$1.50/mi); quotes will test this"
    fig.text(0.5, 0.01, caption, ha="center", fontsize=9)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"{OUTPUT_DIR}/fare_vs_distance.png", dpi=300)
    return fig


fig = plot_fare_vs_distance()

### Quote collection sample (methods figure)

In [ ]:
def plot_quote_sample():
    points = gpd.GeoDataFrame(quotes, geometry=gpd.points_from_xy(quotes["lon"], quotes["lat"]), crs=4326).to_crs(CRS)
    done = points["price"].notna()
    fig, ax = plt.subplots(figsize=(7, 7.5))
    tracts.plot(ax=ax, color="#EEEEEE", edgecolor="white", linewidth=0.2)
    counties.boundary.plot(ax=ax, color="gray", linewidth=0.6)

    for airport in AIRPORTS:
        mine = points["code"] == airport
        color = AIRPORT_COLORS[airport]
        if (mine & done).any():
            points[mine & done].plot(ax=ax, color=color, edgecolor="black", linewidth=0.4, markersize=40, label=f"{airport} collected")
        if (mine & ~done).any():
            points[mine & ~done].plot(ax=ax, facecolor="none", edgecolor=color, linewidth=1.5, markersize=40, label=f"{airport} pending")
        terminals[terminals["code"] == airport].plot(ax=ax, marker="*", color=color, edgecolor="black", markersize=350, zorder=3)

    ax.set_axis_off()
    ax.legend(frameon=False, loc="lower left", fontsize=9)
    fig.suptitle(f"Quote sample: {len(points)} tracts, {done.sum()} collected", fontsize=12)
    caption = "40 tracts per airport, 4 from each distance decile, drawn in proportion to airport workers living there (star = terminal)"
    fig.text(0.5, 0.02, caption, ha="center", fontsize=8.5, wrap=True)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"{OUTPUT_DIR}/quote_sample_map.png", dpi=300)
    return fig


fig = plot_quote_sample()